# CJ-GMP 과제 — LangChain APIs (Google / OpenAI / Anthropic)

이 노트북은 Colab 과제의 `YOUR CODE HERE` 빈칸을 채운 완성본입니다.

- 문제 1: 랩 노래 생성 (창의성 활용)
- 문제 2: 여러 제품 리뷰 요약 (for 루프)
- 문제 3: 범용 번역기 (원어 / 영어 / 한국어)
- 문제 4: 감성 분석 및 객체 식별 (JSON 출력)

## 패키지 설치 및 API 설정

In [ ]:
!python -m pip install -qU langchain 'langchain[google-genai]' 'langchain[openai]' 'langchain[anthropic]' 2> /dev/null
print('--- Job Completed', '-' * 55, '\n')

### Google / OpenAI / Anthropic API Key 설정 (택1)

In [ ]:
# 방법 1 - Colab 보안 비밀에 저장된 API Key 불러오기
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY']    = userdata.get('GOOGLE_API_KEY')
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

In [ ]:
# 방법 2 - 직접 입력
import os, getpass

os.environ['GOOGLE_API_KEY']    = getpass.getpass('Enter Your Google Gemini API Key: ')
os.environ['OPENAI_API_KEY']    = getpass.getpass('Enter Your OpenAI API Key: ')
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter Your Anthropic API Key: ')

## 도우미 함수

In [ ]:
from typing import Any
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.messages import BaseMessage

# 사용할 모델을 본인 환경에 맞게 변경하세요.
# google -> 'google_genai:gemini-2.5-flash' | 'google_genai:gemini-2.5-pro'
# openai -> 'openai:gpt-5' | 'openai:gpt-5-mini' | 'openai:gpt-4o' | 'openai:gpt-4.1'
# anthropic -> 'anthropic:claude-sonnet-4-6' | 'anthropic:claude-opus-4-7'
MODEL = 'google_genai:gemini-2.5-flash'

def get_ai_response(instruction=None, user_message=None, model=MODEL, *, messages=None, **kwargs):
    llm = init_chat_model(model, **kwargs)
    if messages is None:
        messages = []
        if instruction:
            messages.append(SystemMessage(content=instruction))
        messages.append(HumanMessage(content=user_message or ''))
    response = llm.invoke(messages)
    content = response.content
    if isinstance(content, list):
        return ''.join(
            block.get('text', '') for block in content if isinstance(block, dict)
        )
    return content or ''

## 문제 1: 랩 노래 생성

시스템 지시문을 잘 작성하여 창의적인 랩송을 생성한다. `temperature`를 1 이상으로 올려 다양성을 확보한다.

In [ ]:
instruction = '''
당신은 한국어 랩 가사를 쓰는 톱티어 래퍼 겸 작사가입니다.
사용자가 입력한 주제(topic)를 받아, 그 주제에 어울리는 한 곡 분량의 창의적인 한국어 랩 가사를 작성하세요.

[작성 규칙]
1. 곡 구성은 다음 형식을 따릅니다.
   - [Intro] 2~4줄
   - [Verse 1] 8~12줄
   - [Hook] 4~6줄 (반복 후렴, 강한 훅 라인)
   - [Verse 2] 8~12줄
   - [Hook] (위와 동일 또는 살짝 변주)
   - [Bridge] 2~4줄
   - [Outro] 2~3줄
2. 라임(rhyme), 펀치라인, 워드플레이, 비유, 메타포를 적극 사용하세요.
3. 절대 진부한 표현("꿈을 향해 달려가", "열정 가득" 등)에 의존하지 마세요.
4. 주제의 분위기에 맞춰 톤(공격적 / 감성적 / 자신감 / 위트)을 자유롭게 잡되, 한 곡 안에서는 일관되게 유지합니다.
5. 가사 외에 코드/설명/메타정보는 출력하지 않습니다. 섹션 헤더([Verse 1] 등)와 가사만 출력합니다.
6. 한국 힙합 신의 자연스러운 어휘와 리듬감을 살리고, 줄 끝의 라임 짝을 의식적으로 맞추세요.
'''

user_message = input('>>> ')

# 창의성을 위해 temperature를 높여 호출 (Gemini/OpenAI 모두 0~2 범위)
response = get_ai_response(instruction, user_message, temperature=1.2)
print(response)

## 문제 2: 여러 제품 리뷰 요약

In [ ]:
# --- 여러 제품 리뷰 요약하기
instruction = '''
당신의 업무는 전자상거래 사이트에 올라온 제품 리뷰를 짧게 요약하는 것입니다.
리뷰를 최대 30 단어로 요약하세요.
'''

# 하이테크 슬림 노트북
review_1 = '''
최근에 하이테크 슬림 노트북을 구매했습니다.
이 노트북은 정말 가볍고 휴대성이 뛰어납니다.
디자인도 매우 세련되어 보이고, 화면 해상도가 높아서 영화 보기나 문서 작업하기에 아주 좋아요.
특히 배터리 수명이 길어서 하루 종일 충전 걱정 없이 사용할 수 있어서 대만족입니다.
하지만 가격이 조금 높은 편이라서 비용을 고려해야 할 것 같아요.
비슷한 성능을 가진 다른 브랜드의 노트북이 좀 더 저렴할 수도 있으니까요.
배송은 예상보다 빨리 와서 기분이 좋았습니다.
노트북을 받자마자 바로 세팅해서 사용해봤는데, 사용하기 쉽고 빠른 부팅 속도에 만족합니다.
'''

# 스마트워치 리뷰
review_2 = '''
최근에 구입한 스마트워치가 기대에 미치지 못했어요.
배터리 수명이 너무 짧아서 하루에 한 번 이상 충전해야 해요.
터치스크린 반응도 느리고, 때때로 앱이 제대로 동작하지 않는 문제가 있었습니다.
가격 대비 성능이 좋지 않아요.
배송은 빨랐지만, 제품에 대한 전반적인 만족도는 낮습니다.
'''

# 진공 청소기 리뷰
review_3 = '''
이 진공 청소기는 정말 강력하고 사용하기 쉬워요.
무선이라서 어디든지 편하게 가져갈 수 있고, 청소 효율이 좋아요.
배터리 수명도 꽤 괜찮습니다.
다만, 청소기가 조금 무거운 편이라 오래 사용하면 팔이 피곤해질 수 있어요.
가격 대비 성능은 만족스럽습니다.
배송도 빠르고 제품 상태도 완벽해서 좋았습니다.
'''

# 블루투스 헤드폰 리뷰
review_4 = '''
이 블루투스 헤드폰은 소리 품질이 좋지 않아요.
저음이 거의 없고, 때때로 연결이 끊기는 현상이 발생했습니다.
착용감도 불편하고, 귀가 아프게 만드는 경우가 종종 있었어요.
가격이 저렴한 편이지만, 이 제품은 가격만큼의 가치도 없는 것 같습니다.
배송 과정에서 상자가 약간 손상된 채로 도착했어요.
'''

# 전동 칫솔 리뷰
review_5 = '''
이 전동 칫솔은 정말 효과적이에요.
칫솔질이 훨씬 쉬워졌고, 이가 깨끗해진 느낌이 들어요.
여러 모드가 있어서 상황에 따라 조절할 수 있고, 배터리 수명도 길어요.
다만, 헤드 교체 비용이 조금 부담스러울 수 있어요.
가격이 적당하고, 배송도 빨라서 만족합니다.
'''

reviews = [review_1, review_2, review_3, review_4, review_5]

# --- for 문을 사용하여 위 리뷰 5개를 요약한 결과를 출력한다.
for idx, review in enumerate(reviews, start=1):
    summary = get_ai_response(instruction, review, temperature=0)
    print(f'[리뷰 {idx} 요약]')
    print(summary.strip())
    print('-' * 60)

## 문제 3: 범용 번역기

다양한 국가의 언어로 된 입력 문장을 받아 다음 형식으로 출력한다.

```
원어 (언어명): 원본 문장
영어: 영어 번역
한국어: 한국어 번역
```

In [ ]:
instruction = '''
당신은 다국어 번역 전문가입니다.
사용자가 입력하는 문장은 어떤 언어든 될 수 있습니다.
각 입력에 대해 아래 형식 그대로만 정확히 출력하세요. 다른 어떤 문장, 설명, 머리말, 코드블록도 추가하지 마세요.

[출력 형식]
원어 (입력 문장의 언어를 한국어로): <원본 문장 그대로>
영어: <영어로 자연스럽게 번역한 문장>
한국어: <한국어로 자연스럽게 번역한 문장>

[규칙]
1. 첫 번째 줄의 "(입력 문장의 언어)" 자리에는 한국어 언어명을 적습니다.
   예) 프랑스어, 스페인어, 이탈리아어, 러시아어, 베트남어, 독일어, 일본어, 중국어 등
2. "원어" 줄에는 사용자의 원본 문장을 변형 없이 그대로 옮겨 적습니다.
3. "영어"와 "한국어" 줄은 원문의 의미와 뉘앙스를 정확히 살려 자연스럽게 번역합니다.
4. 항상 3줄(원어 / 영어 / 한국어)만 출력합니다. 빈 줄, 추가 설명, 주석은 넣지 않습니다.
'''

user_messages = (
    'La batterie de mon téléphone est trop courte et dure à peine une journée.',
    'La calidad de la cámara de mi teléfono es mala, especialmente en condiciones de poca luz.',
    'Spesso riscontro chiamate perse e scarsa potenza del segnale con il mio telefono.',
    'Сенсорный экран моего телефона не реагирует и иногда не регистрирует касания.',
    'Bản cập nhật phần mềm mới nhất đã làm cho điện thoại của tôi chạy chậm và hay gặp lỗi hơn.'
)
for user_message in user_messages:
    response = get_ai_response(instruction, user_message, temperature=0)
    print(response, '\n')

## 문제 4: 감성 분석 및 객체 식별

리뷰에서 `sentiment`, `satisfaction`(boolean), `item`, `brand`를 추출하여 JSON 형식으로 출력한다.

In [ ]:
instruction = '''
당신은 고객 제품 리뷰를 분석하여 구조화된 정보를 추출하는 분석 엔진입니다.
사용자가 제공하는 리뷰를 읽고, 아래 JSON 스키마에 맞춰 한 개의 JSON 객체만 출력하세요.

[추출 항목]
- sentiment : 리뷰의 전반적 감성. 다음 중 하나의 문자열
    "positive" | "neutral" | "negative"
- satisfaction : 고객이 최종적으로 만족했는지 여부. JSON boolean (true 또는 false).
    반드시 문자열이 아닌 진짜 boolean 값으로 출력합니다.
- item  : 고객이 구매한 제품명 (한국어 문자열)
- brand : 제품의 제조사 또는 브랜드명 (한국어 문자열)

[출력 규칙]
1. 출력은 다음과 같이 ```json 코드 블록 한 개로 감싼 JSON 객체만 출력합니다.
2. JSON 외의 어떤 설명, 머리말, 꼬리말, 추가 문장도 출력하지 않습니다.
3. satisfaction 키의 값은 "true"/"false" 같은 문자열이 아니라 반드시 true / false 인 boolean 이어야 합니다.
4. 정보가 명시되지 않은 경우에도 리뷰 맥락에서 합리적으로 추론합니다.

[출력 형식 예시]
```json
{
  "sentiment": "negative",
  "satisfaction": false,
  "item": "제품명",
  "brand": "브랜드명"
}
```
'''

user_message = '''최근에 키친아트의 스마트케틀 전기 주전자를 구매했는데, 디자인은 괜찮았지만 가격 대비 기대에 못 미쳤어요.
배송은 빨랐지만, 제품을 사용해보니 물이 살짝 새는 문제가 있었습니다.
이 문제를 키친아트에 알렸더니 새 스마트케틀을 보내주겠다고 했지만, 배송이 예상보다 오래 걸렸어요.
새로 온 주전자는 문제가 없었지만, 처음에 문제가 생긴 것이 실망스러웠습니다.
또, 스마트케틀이 조금 시끄러워서 조용한 환경에서는 사용하기가 불편합니다.'''

response = get_ai_response(instruction, user_message, temperature=0)
print(response)